# 🚀 Training ACD-MMKT on Kaggle (2x NVIDIA T4 GPUs)
**Mô hình:** Asymmetric Causal-Disentangled Multimodal Knowledge Tracing (ACD-MMKT)  
**Tập dữ liệu:** Preprocessed XES3G5M (Chinese Mathematics Knowledge Tracing)  
**Mục tiêu:** Tận dụng 2x GPU NVIDIA Tesla T4 (mỗi GPU 16GB VRAM) để huấn luyện tối ưu tốc độ, hỗ trợ chạy song song 5-fold cross-validation hoặc huấn luyện đối chứng Full Model vs Ablation.

---

### ⚙️ Hướng Dẫn Cấu Hình Kaggle Notebook (Quan Trọng):
1. **Accelerator:** Ở thanh menu bên phải (**Session options**), chọn **GPU T4 x 2**.
2. **Internet:** Chuyển sang **Internet ON** (bắt buộc để tải package và clone git).
3. **Data Input:** Nhấn **+ Add Input**, chọn Dataset XES3G5M đã nén/tiền xử lý của bạn (được gắn vào `/kaggle/input/...`).

In [ ]:
# [1] Kiểm tra cấu hình phần cứng & Cấu hình chống phân mảnh VRAM
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!nvidia-smi

import torch
print(f"🔥 PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
gpu_count = torch.cuda.device_count()
print(f"Số lượng GPU nhận diện: {gpu_count}")

for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  [GPU {i}] {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB | Compute Capability: {props.major}.{props.minor}")

assert gpu_count >= 1, "⚠️ Vui lòng chuyển Accelerator sang GPU T4 x 2 trong cài đặt Kaggle!"

In [ ]:
# [2] Thiết lập môi trường làm việc & Clone/Pull repository từ GitHub
import os
import sys

REPO_URL = "https://github.com/nbao0208/kt-mmkt-research.git"
REPO_DIR = "/kaggle/working/kt-mmkt-research"

if os.path.exists("src/models/mmkt/acd_mmkt.py"):
    print("✅ Bạn đang đứng trực tiếp trong thư mục repository kt-mmkt-research.")
    !git pull origin main
elif os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"✅ Đã chuyển thư mục làm việc đến: {REPO_DIR}")
    !git pull origin main
else:
    print(f"📥 Đang clone repository từ {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    print(f"✅ Đã clone thành công và chuyển vào: {os.getcwd()}")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Working Directory:", os.getcwd())

In [ ]:
# [3] Cài đặt & cập nhật các thư viện phụ thuộc (Dependencies)
!pip install -q -r requirements.txt
print("✅ Hoàn tất cài đặt toàn bộ gói phụ thuộc!")

In [ ]:
# [4] Tự động phát hiện và liên kết (Symlink) tập dữ liệu XES3G5M từ /kaggle/input/
import glob
from pathlib import Path

# Tìm thư mục chứa static_item_bank.pt
candidates = glob.glob("/kaggle/input/**/static_item_bank.pt", recursive=True)
if not candidates:
    raise FileNotFoundError(
        "❌ Không tìm thấy static_item_bank.pt trong /kaggle/input/!\n"
        "👉 Vui lòng nhấn nút '+ Add Input' trên Kaggle và thêm Dataset XES3G5M đã tiền xử lý của bạn."
    )

input_data_dir = Path(candidates[0]).parent
print(f"🔍 Đã tìm thấy thư mục chứa dữ liệu tại: {input_data_dir}")

target_dir = Path("data/processed/XES3G5M")
target_dir.mkdir(parents=True, exist_ok=True)

# Tạo symlink cho toàn bộ file .pt và .json
linked_count = 0
for ext in ["*.pt", "*.json"]:
    for src_file in input_data_dir.glob(ext):
        dst_file = target_dir / src_file.name
        if dst_file.exists() or dst_file.is_symlink():
            dst_file.unlink()
        dst_file.symlink_to(src_file)
        linked_count += 1

print(f"✅ Đã tạo symbolic link thành công cho {linked_count} tệp dữ liệu vào {target_dir}.")
print("\n📁 Chi tiết các tệp dữ liệu sẵn sàng cho huấn luyện:")
!ls -lh data/processed/XES3G5M

In [ ]:
# [5] Chạy Sanity Check Unit Tests để đảm bảo kiến trúc và CUDA hoạt động hoàn hảo
!python -m pytest tests/test_acd_mmkt_model.py -v
print("🎉 Toàn bộ 9/9 Unit Tests cho ACD-MMKT đã vượt qua!")

## 🏋️‍♂️ Các Kịch Bản Huấn Luyện Trên Kaggle 2x T4

**Khuyến nghị:**
1. **Chạy Kịch bản 1 (Fold 0):** Huấn luyện Fold 0 với profile `kaggle_t4` (`batch_size=64`, `grad_accum=2` tương đương batch 128) để kiểm tra toàn bộ luồng huấn luyện Giai đoạn 1 & 2 hoạt động mượt mà trên GPU 0 trong ~15-20 phút.
2. **Chạy Kịch bản 2:** Huấn luyện toàn diện 5-Fold Cross-Validation song song trên cả 2 GPU (GPU 0 chạy Folds 0, 1, 2; GPU 1 chạy Folds 3, 4).
3. **Chạy Kịch bản 3:** Huấn luyện đối chứng song song cùng lúc: GPU 0 chạy Full Model, GPU 1 chạy Ablation 2 (w/o Distillation).

In [ ]:
# ==============================================================================
# [Kịch bản 1] Huấn luyện Đơn Fold (Fold 0) với Profile kaggle_t4 trên GPU 0 (cuda:0)
# ==============================================================================
# - Profile kaggle_t4: batch_size=32, grad_accum=4 (batch hiệu dụng 128), AMP FP16
# - Tối ưu VRAM: Micro-chunking tự động chia nhỏ 128 items giúp VRAM luôn ổn định an toàn dưới 8 GB!
# - Giai đoạn 1: 15 epochs (~1 phút)
# - Giai đoạn 2: 30 epochs

!python -u scripts/train_acd_mmkt.py \
    --val-fold 0 \
    --profile kaggle_t4 \
    --device cuda:0 \
    --stage1-epochs 15 \
    --num-epochs 30 \
    --save-dir outputs/checkpoints/acd_mmkt_fold0

In [ ]:
# ==============================================================================
# [Kịch bản 2] Huấn luyện Toàn diện 5-Fold Cross-Validation Song Song Trên 2x GPU T4
# ==============================================================================
# - GPU 1 (CUDA_VISIBLE_DEVICES=1): Đảm nhận Folds [3, 4] | batch_size=64, grad_accum=2, AMP FP16
# - GPU 0 (CUDA_VISIBLE_DEVICES=0): Đảm nhận Folds [0, 1, 2] | batch_size=64, grad_accum=2, AMP FP16
# 🚀 Tách biệt hoàn toàn CUDA Context giữa 2 GPU, VRAM an toàn tuyệt đối và tối ưu 100% 2 GPU!

import subprocess
import time
import os
import sys

os.makedirs("outputs/logs", exist_ok=True)
log_gpu1_path = "outputs/logs/worker_gpu1.log"

# Môi trường biệt lập cho từng GPU
env_gpu1 = os.environ.copy()
env_gpu1["CUDA_VISIBLE_DEVICES"] = "1"
env_gpu1["PYTHONUNBUFFERED"] = "1"
env_gpu1["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

env_gpu0 = os.environ.copy()
env_gpu0["CUDA_VISIBLE_DEVICES"] = "0"
env_gpu0["PYTHONUNBUFFERED"] = "1"
env_gpu0["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("🚀 [1/2] Đang kích hoạt Worker ngầm trên GPU 1 cho Folds [3, 4]...")
cmd_gpu1 = [
    sys.executable, "-u", "scripts/run_5fold_acd_mmkt.py",
    "--device", "cuda:0",
    "--folds", "3", "4",
    "--batch-size", "32",
    "--grad-accum-steps", "4",
    "--precision", "amp_fp16",
    "--num-epochs", "30",
    "--checkpoint-dir", "outputs/checkpoints",
    "--worker-only"
]

f_log1 = open(log_gpu1_path, "w")
proc_gpu1 = subprocess.Popen(cmd_gpu1, stdout=f_log1, stderr=subprocess.STDOUT, env=env_gpu1)
print(f"✅ Worker GPU 1 đang khởi tạo (PID: {proc_gpu1.pid}). Log ghi trực tiếp tại: {log_gpu1_path}")

# Giãn cách 10 giây để GPU 1 nạp xong mô hình
print("⏳ Giãn cách 10s để GPU 1 nạp xong mô hình tránh tranh chấp đĩa...")
time.sleep(10)

print("\n🚀 [2/2] Đang khởi chạy Worker trên GPU 0 cho Folds [0, 1, 2] trên tiến trình chính...")
cmd_gpu0 = [
    sys.executable, "-u", "scripts/run_5fold_acd_mmkt.py",
    "--device", "cuda:0",
    "--folds", "0", "1", "2",
    "--batch-size", "32",
    "--grad-accum-steps", "4",
    "--precision", "amp_fp16",
    "--num-epochs", "30",
    "--checkpoint-dir", "outputs/checkpoints",
    "--worker-only"
]
proc_gpu0 = subprocess.Popen(cmd_gpu0, env=env_gpu0)

# Chờ 2 tiến trình
proc_gpu0.wait()
f_log1.flush()
proc_gpu1.wait()
f_log1.close()

print(f"\n✅ Cả 2 GPU đã hoàn thành 5 Folds! (Exit codes: GPU0={proc_gpu0.returncode}, GPU1={proc_gpu1.returncode})")

print("\n📊 Đang tổng hợp báo cáo đánh giá trung bình 5-Fold (Mean ± Std)...\n")
!python -u scripts/run_5fold_acd_mmkt.py --eval-only --folds 0 1 2 3 4 --checkpoint-dir outputs/checkpoints

In [ ]:
# ==============================================================================
# [Kịch bản 3] Huấn luyện Song Song Đối Chứng: Full Model (GPU 0) vs Ablation 2 (GPU 1)
# ==============================================================================
import subprocess
import os
import sys
import time

os.makedirs("outputs/logs", exist_ok=True)
log_ablation = "outputs/logs/ablation2_gpu1.log"

env_gpu1 = os.environ.copy()
env_gpu1["CUDA_VISIBLE_DEVICES"] = "1"
env_gpu1["PYTHONUNBUFFERED"] = "1"
env_gpu1["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

env_gpu0 = os.environ.copy()
env_gpu0["CUDA_VISIBLE_DEVICES"] = "0"
env_gpu0["PYTHONUNBUFFERED"] = "1"
env_gpu0["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("🚀 [1/2] Đang chạy Ablation 2 (w/o Distillation) trên GPU 1...")
cmd_ablation = [
    sys.executable, "-u", "scripts/train_acd_mmkt.py",
    "--val-fold", "0",
    "--ablation", "2",
    "--device", "cuda:0",
    "--precision", "amp_fp16",
    "--batch-size", "32",
    "--grad-accum-steps", "4",
    "--num-workers", "2",
    "--num-epochs", "30",
    "--save-dir", "outputs/checkpoints/ablation2_wo_distill"
]
f_abl = open(log_ablation, "w")
proc_abl = subprocess.Popen(cmd_ablation, stdout=f_abl, stderr=subprocess.STDOUT, env=env_gpu1)

time.sleep(10)

print("🚀 [2/2] Đang chạy Full ACD-MMKT Model trên GPU 0...")
cmd_full = [
    sys.executable, "-u", "scripts/train_acd_mmkt.py",
    "--val-fold", "0",
    "--ablation", "1",
    "--device", "cuda:0",
    "--precision", "amp_fp16",
    "--batch-size", "32",
    "--grad-accum-steps", "4",
    "--num-workers", "2",
    "--num-epochs", "30",
    "--save-dir", "outputs/checkpoints/acd_mmkt_fold0"
]
proc_full = subprocess.Popen(cmd_full, env=env_gpu0)

proc_full.wait()
f_abl.flush()
proc_abl.wait()
f_abl.close()
print("✅ Cả 2 mô hình đã hoàn thành huấn luyện trên 2 GPU!")

In [ ]:
# [6] Đánh giá chi tiết trên tập Test (1.099.197 tương tác) & Báo cáo giải thích Ante-hoc
!python -u scripts/evaluate_acd_mmkt.py \
    --checkpoint outputs/checkpoints/acd_mmkt_fold0/best_model.pt \
    --val-fold 0 \
    --device cuda:0 \
    --output-report outputs/reports/eval_fold0_detailed.json \
    --explain-output-md outputs/reports/explainability_sample.md

import json
import pandas as pd

report_file = "outputs/reports/eval_fold0_detailed.json"
if os.path.exists(report_file):
    with open(report_file) as f:
        res = json.load(f)
    print("\n🏆 BẢNG TỔNG HỢP KẾT QUẢ ĐÁNH GIÁ (GLOBAL MICRO ROC-AUC & ACC):")
    summary_df = pd.DataFrame([{
        "Overall AUC": f"{res.get('global_metrics', {}).get('auc', res.get('auc', 0)):.4f}",
        "Overall ACC": f"{res.get('global_metrics', {}).get('acc', res.get('acc', 0)):.4f}",
        "Pearson r (CTT)": f"{res.get('psychometric_correlation', {}).get('item_bank_intrinsic', {}).get('pearson_r', res.get('psychometrics', {}).get('pearson_r', 0)):.4f}",
        "Spearman rho (CTT)": f"{res.get('psychometric_correlation', {}).get('item_bank_intrinsic', {}).get('spearman_rho', res.get('psychometrics', {}).get('spearman_rho', 0)):.4f}"
    }])
    display(summary_df)
else:
    print("Không tìm thấy file báo cáo.")

In [ ]:
# [7] Đóng gói toàn bộ Checkpoints, Reports và Logs thành file ZIP để tải về máy
!zip -q -r /kaggle/working/acd_mmkt_results.zip outputs/checkpoints outputs/reports outputs/logs
print("🎉 Đã nén thành công toàn bộ kết quả vào: /kaggle/working/acd_mmkt_results.zip")
print("👉 Bạn có thể tải file này trực tiếp từ khung 'Output' bên phải giao diện Kaggle!")